# Join datatables to be used for GLMM 
- information about elements, genomic windows, chromosomes, species
Joins element-level data with metadata, chromosome sizes, and window-level (size 5%, step 1%) summaries to create data/df_for_linear_models.tsv — the input for GLMM analysis.
Input: Master element tables, metadata.tsv, chromosome_sizes.tsv, solo-intact_ratios.tsv
Output: data/df_for_linear_models.tsv


In [17]:
import pandas as pd

# Load basic element tables

In [18]:
intact_df = pd.read_csv(f"data/intact_elements_dante.tsv", sep="\t")
partial_df = pd.read_csv(f"data/partial_elements_dante.tsv", sep="\t")
solo_df = pd.read_csv(f"data/solo_elements.tsv", sep="\t")

# Add a column to identify the type of element
intact_df['element_type'] = 'intact'
partial_df['element_type'] = 'partial'
solo_df['element_type'] = 'solo_LTR'


# Prepare the solo LTR table

In [19]:
solo_columns = ['chr', 'start', 'end', 'ID', 'origin_element_ID', 'origin_LTR_coverage', 'sample', 'drop', 'Class', 'element_type']
solo_df.columns = solo_columns
# drop drop column
solo_df = solo_df.drop(columns=['drop'])

# Join element types into one table

In [20]:
df_unmerged = pd.concat([intact_df, partial_df, solo_df], ignore_index=True)

# Add metadata

In [21]:
metadata_df = pd.read_csv(f"data/metadata.tsv", sep="\t")
print("Metadata columns:", metadata_df.columns.tolist())
metadata_columns = ['Order',
 'Species',
 'Chromosome_number_(n)',
 'Assembly_size_(Mb)',
 'Chromosome_size_(Mb)',
 'Chromosomes_in_analysis',
 'Assembly_size_in_analysis_(Mb)',
 'Accession',
 'Family',
 'Centromere_architecture',
 'Included_chromosomes',
 'Gaps_per_chr',
 'Gaps_per_Mb',
 'Busco_pct',
 'Short_label']
metadata_df.columns = metadata_columns


Metadata columns: ['Order', 'Species', 'Chromosome number (n)', 'Assembly size (Mb)', 'Chromosome size (Mb)', 'Chromosomes in analysis', 'Assembly size in analysis (Mb)', 'Accession', 'Family', 'Centromere architecture', 'Excluded (-) or included (+) chr IDs', 'Gaps per chromosome', 'Gaps per Mb', 'BUSCO complete (%)', 'Short label']


In [32]:
# add Genus, Order and Clade columns to metadata
family_to_order = {
    'Poaceae': 'Poales',
    'Cyperaceae': 'Poales',
    'Juncaceae': 'Poales',
    'Convolvulaceae': 'Solanales',
    'Melanthiaceae': 'Liliales'}

metadata_df['Ordo'] = metadata_df['Family'].map(family_to_order)

order_to_clade = {
    'Poales': 'Monocots',
    'Solanales': 'Eudicots',
    'Liliales': 'Monocots'}

metadata_df['Clade'] = metadata_df['Ordo'].map(order_to_clade)

metadata_df['Genus'] = metadata_df['Species'].str.split("_").str[0]    

In [33]:
df_metadata = df_unmerged.merge(metadata_df, left_on='sample', right_on='Species', how='left')
df_metadata.head()

,ID,sample,chr,start,end,Class,Rank,LTR_identity,LTR_length,element_type,...,Family,Centromere_architecture,Included_chromosomes,Gaps_per_chr,Gaps_per_Mb,Busco_pct,Short_label,Ordo,Clade,Genus
0,TE_00000001_CM062751.1,Bolboschoenus_planiculmis,CM062751.1,258457,263448,Ty1/copia|Ivana,DLP,96.121,462.0,intact,...,Cyperaceae,holocentric,-,0.37,0.08,75.6,Bo. planiculmis,Poales,Monocots,Bolboschoenus
1,TE_00000002_CM062751.1,Bolboschoenus_planiculmis,CM062751.1,4327653,4333370,Ty3/gypsy|chromovirus|CRM,DLTP,98.645,664.0,intact,...,Cyperaceae,holocentric,-,0.37,0.08,75.6,Bo. planiculmis,Poales,Monocots,Bolboschoenus
2,TE_00000003_CM062751.1,Bolboschoenus_planiculmis,CM062751.1,4390103,4400733,Ty3/gypsy|non-chromovirus|OTA|Athila,DL,91.902,1327.0,intact,...,Cyperaceae,holocentric,-,0.37,0.08,75.6,Bo. planiculmis,Poales,Monocots,Bolboschoenus
3,TE_00000004_CM062751.1,Bolboschoenus_planiculmis,CM062751.1,4424594,4431916,Ty1/copia|Ikeros,DLTP,99.242,660.0,intact,...,Cyperaceae,holocentric,-,0.37,0.08,75.6,Bo. planiculmis,Poales,Monocots,Bolboschoenus
4,TE_00000001_CM062752.1,Bolboschoenus_planiculmis,CM062752.1,626061,631038,Ty1/copia|Ivana,DLTP,100.000,456.0,intact,...,Cyperaceae,holocentric,-,0.37,0.08,75.6,Bo. planiculmis,Poales,Monocots,Bolboschoenus


# Add chromosome sizes


In [40]:
chrom_sizes_df = pd.read_csv(f"data/chromosome_sizes.tsv", sep="\t")
df_chrom_sizes = df_metadata.merge(chrom_sizes_df, left_on='chr', right_on='chr', how='left')
df_chrom_sizes.head()
# number of NAs in chrom_size column

print("Number of NAs in chrom_size column:", df_chrom_sizes['length'].isna().sum())

# remove chr lengths from Chamaelirium luteum because they are not actually chromosome sizes but scaffold sizes
df_chrom_sizes.loc[df_chrom_sizes['sample'] == 'Chamaelirium_luteum', 'length'] = pd.NA


Number of NAs in chrom_size column: 0


# Add genomic windows

In [41]:
window_df = pd.read_csv(f"data/solo-intact_ratios.tsv", sep="\t")
print("100kb window columns:", window_df.columns.tolist())

100kb window columns: ['sample', 'chr', 'bin_ID', 'bin_start', 'bin_end', 'all_solo-intact_ratio', 'all_intact_LTR_identity', 'all_intact_count', 'all_disrupted_count', 'all_solo_count']


In [42]:
window_columns = ['sample', 'chr', 'bin_ID', 'bin_start', 'bin_end', 'all_solo-intact_ratio_in_window', 'all_intact_LTR_identity_in_window', 'all_intact_count_in_window', 'all_disrupted_count_in_window', 'all_solo_count_in_window']
window_df.columns = window_columns

In [43]:
df_with_windows = []

for (sample, chr_name), group in df_chrom_sizes.groupby(['sample', 'chr']):
    windows_group = window_df[(window_df['sample'] == sample) & 
                               (window_df['chr'] == chr_name)].copy()
    
    if len(windows_group) == 0:
        df_with_windows.append(group)
        continue
    
    # Sort both dataframes
    group_sorted = group.sort_values('start').copy()
    windows_sorted = windows_group.sort_values('bin_start').copy()
    
    # Merge using merge_asof
    merged = pd.merge_asof(
        group_sorted,
        windows_sorted,
        left_on='start',
        right_on='bin_start',
        suffixes=('', '_window'),
        direction='backward'
    )
    
    # Filter to keep only elements that actually fall within the window
    merged = merged[
        (merged['start'] >= merged['bin_start']) & 
        (merged['start'] < merged['bin_end'])
    ]
    
    df_with_windows.append(merged)

df = pd.concat(df_with_windows, ignore_index=True)

In [44]:
df.columns
new_names = [['ID', 'sample', 'chr', 'start', 'end', 'Class', 'Rank', 'LTR_identity',
       'LTR_length', 'element_type', 'origin_element_ID',
       'origin_LTR_coverage', 'Order', 'Species', 'Chromosome_number_(n)',
       'Assembly_size_(Mb)', 'Avg_Chromosome_size_(Mb)', 'Chromosomes_in_analysis',
       'Assembly_size_in_analysis_(Mb)', 'Accession', 'Family',
       'Centromere_architecture', 'Included_chromosomes', 'Gaps_per_chr',
       'Gaps_per_Mb', 'Busco_pct', 'Short_label', 'Ordo', 'Clade', 'Genus', 'species',
       'chr_length', 'sample_window', 'chr_window', 'bin_ID', 'bin_start',
       'bin_end', 'all_solo-intact_ratio_in_window',
       'all_intact_LTR_identity_in_window', 'all_intact_count_in_window',
       'all_disrupted_count_in_window', 'all_solo_count_in_window']]

drop = ["sample", "Accession", "Included_chromosomes", "species", "sample_window", "chr_window"]
df = df.rename(columns=dict(zip(df.columns, new_names[0]))).drop(columns=drop)

In [45]:
# save the final dataframe to a tsv file
df.to_csv(f"data/df_for_linear_models.tsv", sep="\t", index=False)